<a href="https://colab.research.google.com/github/energetik23/Python_1sem/blob/main/5_%D0%BF%D1%80%D0%B0%D0%BA%D1%82%D0%B8%D1%87%D0%B5%D1%81%D0%BA%D0%B0%D1%8F.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. Используя данные из файла students.json посчитать средний балл студентов, общий средний балл и сохранить отчет в report.json. Обработать ошибки с отсутствием файла, отсутствием ключей name и grades, неверный тип данных.
Структура файла:
[
  {
    'name': 'Anna',
    'grades': [5, 3, 4, 3],
  },
  {
    'name': 'Ivan',
    'grades': [5, 4, 4,  2],
  },
  {
    'name': 'Sergey',
    'grades': [3, 5, 4, 5],
  },
  {
    'name': 'Maria',
    'grades': [4, 3, 2, 4],
  },
]

In [ ]:
import json
import os


def load_students(filename):
    """Загружю список студентов из JSON файла."""
    if not os.path.exists(filename):
        raise FileNotFoundError(f"Файл '{filename}' не найден")

    with open(filename, "r", encoding="utf-8") as f:
        data = json.load(f)

    if not isinstance(data, list):
        raise ValueError("JSON должен быть списком студентов")

    return data


def validate_student(student, index):
    """Проверяю 1 одного студента и возвращаю (name, grades)."""
    if not isinstance(student, dict):
        raise ValueError(f"Студент #{index}: ожидался объект, получено {type(student).__name__}")

    if "name" not in student:
        raise KeyError(f"Студент #{index}: отсутствует ключ 'name'")
    if "grades" not in student:
        raise KeyError(f"Студент #{index}: отсутствует ключ 'grades'")

    name = student["name"]
    grades = student["grades"]

    if not isinstance(name, str) or not name.strip():
        raise TypeError(f"Студент #{index}: 'name' должен быть непустой строкой")

    if not isinstance(grades, list):
        raise TypeError(f"Студент #{index} ({name}): 'grades' должен быть списком")

    for g in grades:
        if not isinstance(g, (int, float)) or isinstance(g, bool):
            raise TypeError(f"Студент #{index} ({name}): оценка '{g}' не является числом")
        if not (1 <= g <= 5):
            raise ValueError(f"Студент #{index} ({name}): оценка {g} вне диапазона 1..5")

    if len(grades) == 0:
        raise ValueError(f"Студент #{index} ({name}): список оценок пуст")

    return name, grades


def build_report(students):
    """Считаю ср баллы и формирую отчёт."""
    per_student = []
    all_grades = []

    for i, student in enumerate(students):
        try:
            name, grades = validate_student(student, i)
        except (KeyError, TypeError, ValueError) as e:
            # Пропускаем некорректного студента, сообщаем об ошибке
            print(f"[ПРОПУСК] {e}")
            continue

        avg = sum(grades) / len(grades)
        per_student.append({
            "name": name,
            "grades": grades,
            "average": round(avg, 2),
        })
        all_grades.extend(grades)

    if not all_grades:
        raise ValueError("Не удалось вычислить средний балл: нет валидных данных")

    overall = sum(all_grades) / len(all_grades)
    return {
        "students": per_student,
        "overall_average": round(overall, 2),
    }


def save_report(report, filename):
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(report, f, ensure_ascii=False, indent=2)


def main():
    try:
        students = load_students("students.json")
    except FileNotFoundError as e:
        print(f"Ошибка: {e}")
        return
    except json.JSONDecodeError as e:
        print(f"Ошибка: файл содержит некорректный JSON ({e})")
        return

    try:
        report = build_report(students)
    except ValueError as e:
        print(f"Ошибка: {e}")
        return

    save_report(report, "report.json")

    print("\n=== Отчёт ===")
    for s in report["students"]:
        print(f"{s['name']:<10} {s['grades']}  средний = {s['average']}")
    print(f"\nОбщий средний балл: {report['overall_average']}")
    print("Отчёт сохранён в report.json")


if __name__ == "__main__":
    main()